# OCR Extractor MVP Demo

This notebook demonstrates the local OCR extraction pipeline for an industrial inspection document. The tool accepts an image path, runs EasyOCR locally, extracts deterministic equipment fields, validates them conservatively, and returns a JSON-style result.

## Expected result
The test document is designed to produce values similar to:

- equipment_id = 11-V-102
- equipment_name = HP Separator Drum
- material = 2.25Cr-1Mo
- design_pressure_mpa = 14.5
- measured_thickness_mm = 138.2
- corrosion_rate_mm_yr = 0.75

In [ ]:
from pathlib import Path
from ocr_extractor.ocr_extractor_tool import ocr_extractor_tool, create_synthetic_inspection_image

project_root = Path.cwd().resolve()
sample_dir = project_root / 'sample_data'
sample_dir.mkdir(parents=True, exist_ok=True)

normal_image = create_synthetic_inspection_image(sample_dir / 'inspection_report_test.png')
missing_field_image = create_synthetic_inspection_image(sample_dir / 'inspection_report_missing_field.png', missing_field='Equipment Name')
pressure_conversion_image = create_synthetic_inspection_image(sample_dir / 'inspection_report_pressure_bar.png', pressure_text='145 bar')

print(normal_image)
print(missing_field_image)
print(pressure_conversion_image)

In [ ]:
normal_result = ocr_extractor_tool(str(normal_image))
print(normal_result)

In [ ]:
missing_result = ocr_extractor_tool(str(missing_field_image))
print(missing_result)

In [ ]:
pressure_result = ocr_extractor_tool(str(pressure_conversion_image))
print(pressure_result)

In [ ]:
assert normal_result['equipment_id'] == '11-V-102'
assert normal_result['equipment_name'] == 'HP Separator Drum'
assert normal_result['material'] == '2.25Cr-1Mo'
assert abs(normal_result['design_pressure_mpa'] - 14.5) < 0.5
assert abs(normal_result['measured_thickness_mm'] - 138.2) < 0.5
assert abs(normal_result['corrosion_rate_mm_yr'] - 0.75) < 0.2
assert missing_result['equipment_name'] is None
assert abs(pressure_result['design_pressure_mpa'] - 14.5) < 0.5
print('Test 1 passed: normal document extraction')
print('Test 2 passed: missing field becomes None')
print('Test 3 passed: bar-to-MPa conversion is correct')
print('All tests passed successfully.')
print('MODULE_TESTS_PASSED')